# 🔍 Advanced RAG Pipeline with OpenAI
**Retrieval-Augmented Generation — Kapsamlı Notebook**

Bu notebook sırasıyla şunları kapsar:
1. Kurulum ve OpenAI bağlantısı
2. Doküman yükleme ve seçimi (PDF, TXT, DOCX, URL)
3. Akıllı chunking stratejileri
4. Embedding ve vektör veritabanı (FAISS)
5. **Advanced teknikler:** HyDE · Query Expansion · Reranking · Parent-Child · Hybrid Search
6. RAG pipeline entegrasyonu
7. **Performans değerlendirmesi:** LLM-as-Judge metrikleri + görsel analiz

---
> ⚙️ **Gereksinim:** OpenAI API anahtarı. Google Colab'da `Secrets` bölümüne `OPENAI_API_KEY` ekleyin.

## 📦 1. Kurulum

In [ ]:
!pip install -q \
    openai \
    faiss-cpu \
    tiktoken \
    pypdf \
    python-docx \
    requests \
    beautifulsoup4 \
    datasets \
    pandas \
    matplotlib \
    seaborn \
    tqdm \
    rank-bm25 \
    ipywidgets

print('✅ Tüm paketler kuruldu.')

In [ ]:
import os, io, re, json, time, textwrap
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

import openai
import tiktoken
import faiss

# Colab Secrets'dan API anahtarı oku
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    raise ValueError('❌ OPENAI_API_KEY bulunamadi! Colab Secrets a ekleyin.')

client = openai.OpenAI(api_key=OPENAI_API_KEY)

EMBED_MODEL  = 'text-embedding-3-small'
CHAT_MODEL   = 'gpt-4o-mini'
RERANK_MODEL = 'gpt-4o-mini'

print(f'✅ OpenAI bağlantısı kuruldu.')
print(f'   Embedding : {EMBED_MODEL}')
print(f'   Chat      : {CHAT_MODEL}')

## 📄 2. Doküman Yükleme ve Seçimi
PDF, TXT, DOCX veya URL girebilirsiniz.

In [ ]:
import urllib.request
from bs4 import BeautifulSoup
import pypdf
import docx

class DocumentLoader:
    """PDF, TXT, DOCX ve URL'den metin yukler."""

    @staticmethod
    def from_pdf(path: str) -> str:
        reader = pypdf.PdfReader(path)
        return '\n'.join(p.extract_text() or '' for p in reader.pages)

    @staticmethod
    def from_txt(path: str) -> str:
        return Path(path).read_text(encoding='utf-8', errors='ignore')

    @staticmethod
    def from_docx(path: str) -> str:
        doc = docx.Document(path)
        return '\n'.join(p.text for p in doc.paragraphs)

    @staticmethod
    def from_url(url: str) -> str:
        with urllib.request.urlopen(url, timeout=10) as resp:
            soup = BeautifulSoup(resp.read(), 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer']):
            tag.decompose()
        return ' '.join(soup.get_text(' ', strip=True).split())

DEMO_TEXT = '''
Retrieval-Augmented Generation (RAG) is a technique that combines large language models
with external knowledge retrieval. Instead of relying solely on the model parametric knowledge,
RAG retrieves relevant documents from a knowledge base and uses them as context for generation.

The RAG pipeline consists of several key components:
1. Document ingestion and chunking
2. Embedding generation using models like text-embedding-3-small
3. Vector storage in databases like FAISS, Pinecone, or Chroma
4. Semantic retrieval using cosine similarity or dot product
5. Context augmented generation with an LLM

Advanced RAG techniques include:
- Hypothetical Document Embeddings (HyDE): Generate a hypothetical answer, embed it, and use
  that embedding for retrieval instead of the original query. This works because answer text
  is semantically closer to document text than question text.
- Query Expansion: Generate multiple paraphrases of the query to improve recall. Combine
  results using Reciprocal Rank Fusion (RRF) to deduplicate and rank.
- Parent-Child Retrieval: Store small chunks for precise matching but return larger parent
  chunks as context for the LLM, providing more coherent information.
- Cross-encoder Reranking: Use a more powerful model to rerank retrieved documents.
- Hybrid Search: Combine dense vector search with sparse BM25 keyword search.

Evaluation of RAG systems uses metrics such as:
- Faithfulness: Does the answer stay faithful to the retrieved context?
- Answer Relevancy: Is the answer relevant to the question?
- Context Precision: Are the retrieved chunks actually relevant to the question?
- Context Recall: Are all relevant documents successfully retrieved?

OpenAI provides text-embedding-3-small and text-embedding-3-large models for embeddings.
The smaller model offers a good balance of performance and cost at 1536 dimensions.
Both support dimension reduction via the dimensions parameter.

Chunking strategies significantly affect RAG performance. Fixed-size chunking divides text
into equal-sized pieces with overlap. Semantic chunking groups sentences by meaning.
Recursive character splitting tries natural boundaries: paragraphs, sentences, then words.
'''

print('✅ DocumentLoader hazır. Demo metin yüklendi.')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

out = widgets.Output()

source_type = widgets.RadioButtons(
    options=['Dosya Yukle (PDF/TXT/DOCX)', 'URL', 'Metin Yapistir', 'Demo Metin (RAG Docs)'],
    description='Kaynak:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)
uploader  = widgets.FileUpload(accept='.pdf,.txt,.docx', multiple=False,
                                layout=widgets.Layout(display='none'))
url_input = widgets.Text(placeholder='https://...',
                          layout=widgets.Layout(width='80%', display='none'))
text_area = widgets.Textarea(placeholder='Metni buraya yapistirin...',
                              layout=widgets.Layout(width='100%', height='150px', display='none'))
load_btn  = widgets.Button(description='Dokumani Yukle', button_style='primary', icon='download')
doc_status = widgets.HTML(value='<i>Henuz dokuman yuklenmedi.</i>')

RAW_DOCUMENTS: List[Dict] = []

def on_source_change(change):
    v = change['new']
    uploader.layout.display  = 'flex' if 'Dosya'  in v else 'none'
    url_input.layout.display = 'flex' if 'URL'    in v else 'none'
    text_area.layout.display = 'flex' if 'Metin'  in v else 'none'

source_type.observe(on_source_change, names='value')

def on_load(b):
    global RAW_DOCUMENTS
    with out:
        clear_output()
        v = source_type.value
        try:
            if 'Dosya' in v:
                if not uploader.value:
                    print('Lutfen bir dosya secin.'); return
                fname, fdata = list(uploader.value.items())[0]
                ext = Path(fname).suffix.lower()
                tmp = f'/tmp/{fname}'
                with open(tmp, 'wb') as f:
                    f.write(fdata['content'])
                if ext == '.pdf':  raw = DocumentLoader.from_pdf(tmp)
                elif ext == '.docx': raw = DocumentLoader.from_docx(tmp)
                else: raw = DocumentLoader.from_txt(tmp)
                source = fname
            elif 'URL' in v:
                raw = DocumentLoader.from_url(url_input.value.strip())
                source = url_input.value.strip()
            elif 'Metin' in v:
                raw = text_area.value
                source = 'Manuel metin'
            else:
                raw = DEMO_TEXT
                source = 'Demo — RAG Docs'

            RAW_DOCUMENTS = [{'source': source, 'text': raw}]
            doc_status.value = f"<b style='color:green'>Yuklendi:</b> {source} — {len(raw):,} karakter"
            print(f'Dokuman yuklendi: {len(raw):,} karakter')
            print(f'Ilk 300 karakter: {raw[:300]}...')
        except Exception as e:
            doc_status.value = f"<b style='color:red'>Hata:</b> {e}"
            print(f'Hata: {e}')

load_btn.on_click(on_load)
display(widgets.VBox([source_type, uploader, url_input, text_area, load_btn, doc_status, out]))

## ✂️ 3. Chunking Stratejileri

In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o')

def count_tokens(text: str) -> int:
    return len(enc.encode(text))


class ChunkingStrategy:

    @staticmethod
    def fixed_size(text: str, chunk_size: int = 512, overlap: int = 64) -> List[Dict]:
        """Sabit token boyutunda ortüşen parcalar."""
        tokens = enc.encode(text)
        chunks = []
        for i in range(0, len(tokens), chunk_size - overlap):
            toks = tokens[i: i + chunk_size]
            chunks.append({'text': enc.decode(toks), 'chunk_id': len(chunks),
                           'strategy': 'fixed_size', 'token_count': len(toks), 'start_token': i})
        return chunks

    @staticmethod
    def recursive_character(text: str, chunk_size: int = 1500, overlap: int = 200) -> List[Dict]:
        """Paragraf sonra cumle sonra kelime sirasıyla böler."""
        separators = ['\n\n', '\n', '. ', ' ', '']

        def split_rec(txt, seps):
            if not seps or len(txt) <= chunk_size: return [txt]
            parts = txt.split(seps[0])
            result, current = [], ''
            for part in parts:
                candidate = current + seps[0] + part if current else part
                if len(candidate) <= chunk_size:
                    current = candidate
                else:
                    if current: result.extend(split_rec(current, seps[1:]))
                    current = part
            if current: result.extend(split_rec(current, seps[1:]))
            return result

        raw_chunks = split_rec(text, separators)
        chunks, buf = [], ''
        for rc in raw_chunks:
            if len(buf) + len(rc) + 1 <= chunk_size:
                buf = buf + ' ' + rc if buf else rc
            else:
                if buf: chunks.append(buf)
                buf = rc
        if buf: chunks.append(buf)
        return [{'text': c.strip(), 'chunk_id': i, 'strategy': 'recursive',
                 'token_count': count_tokens(c), 'start_token': -1}
                for i, c in enumerate(chunks)]

    @staticmethod
    def parent_child(text: str, parent_size: int = 2000,
                      child_size: int = 400, overlap: int = 50):
        """Buyuk parent + kucuk child chunk lar doner."""
        parents = ChunkingStrategy.fixed_size(text, parent_size, overlap=100)
        children = []
        for p in parents:
            kids = ChunkingStrategy.fixed_size(p['text'], child_size, overlap)
            for k in kids:
                k['parent_id'] = p['chunk_id']
                k['chunk_id']  = len(children)
                k['strategy']  = 'child'
                children.append(k)
        return parents, children


# Widget
strategy_dd = widgets.Dropdown(
    options=[('Fixed Size (512 token)', 'fixed'),
             ('Recursive Character Split', 'recursive'),
             ('Parent-Child (Advanced)', 'parent_child')],
    description='Strateji:', style={'description_width': 'initial'}
)
chunk_btn = widgets.Button(description='Chunkla', button_style='success', icon='scissors')
chunk_out = widgets.Output()

CHUNKS: List[Dict] = []
PARENT_CHUNKS: List[Dict] = []

def on_chunk(b):
    global CHUNKS, PARENT_CHUNKS
    with chunk_out:
        clear_output()
        if not RAW_DOCUMENTS:
            print('Once dokuman yukleyin!'); return
        text     = ' '.join(d['text'] for d in RAW_DOCUMENTS)
        strategy = strategy_dd.value
        if strategy == 'fixed':
            CHUNKS = ChunkingStrategy.fixed_size(text)
            PARENT_CHUNKS = []
        elif strategy == 'recursive':
            CHUNKS = ChunkingStrategy.recursive_character(text)
            PARENT_CHUNKS = []
        else:
            PARENT_CHUNKS, CHUNKS = ChunkingStrategy.parent_child(text)
            print(f'Parent chunk sayisi: {len(PARENT_CHUNKS)}')
        token_counts = [c['token_count'] for c in CHUNKS]
        print(f'Chunking tamamlandi ({strategy})')
        print(f'Chunk sayisi        : {len(CHUNKS)}')
        print(f'Ort. token/chunk    : {np.mean(token_counts):.0f}')
        print(f'Min / Max           : {min(token_counts)} / {max(token_counts)}')
        print(f'\nOrnek chunk (ilk): {CHUNKS[0]["text"][:300]}...')

chunk_btn.on_click(on_chunk)
display(widgets.HBox([strategy_dd, chunk_btn]))
display(chunk_out)

## 🧮 4. Embedding ve FAISS Vektör Deposu

In [ ]:
def get_embeddings(texts: List[str], model: str = EMBED_MODEL,
                   batch_size: int = 100) -> np.ndarray:
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = texts[i: i + batch_size]
        resp  = client.embeddings.create(input=batch, model=model)
        all_embs.extend([e.embedding for e in resp.data])
        time.sleep(0.2)
    return np.array(all_embs, dtype='float32')


class FAISSVectorStore:
    """FAISS tabanli vektor deposu — cosine similarity."""

    def __init__(self, dim: int = 1536):
        self.dim   = dim
        self.index = faiss.IndexFlatIP(dim)
        self.chunks: List[Dict] = []

    def add(self, chunks: List[Dict], embeddings: np.ndarray):
        norms  = np.linalg.norm(embeddings, axis=1, keepdims=True)
        normed = embeddings / (norms + 1e-10)
        self.index.add(normed)
        self.chunks.extend(chunks)

    def search(self, query_emb: np.ndarray, k: int = 5) -> List[Dict]:
        q = query_emb / (np.linalg.norm(query_emb) + 1e-10)
        scores, idxs = self.index.search(q.reshape(1, -1), k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx >= 0:
                r = dict(self.chunks[idx])
                r['score'] = float(score)
                results.append(r)
        return results

    @property
    def size(self): return self.index.ntotal

print('✅ FAISSVectorStore hazir.')

In [ ]:
VECTOR_STORE: Optional[FAISSVectorStore] = None
BM25_INDEX = None

if not CHUNKS:
    print('Once chunking yapın (Hucre 3).')
else:
    texts = [c['text'] for c in CHUNKS]
    print(f'{len(texts)} chunk icin embedding uretiliyor...')
    embeddings = get_embeddings(texts)
    dim = embeddings.shape[1]
    VECTOR_STORE = FAISSVectorStore(dim=dim)
    VECTOR_STORE.add(CHUNKS, embeddings)

    PARENT_STORE = None
    if PARENT_CHUNKS:
        p_embs = get_embeddings([c['text'] for c in PARENT_CHUNKS])
        PARENT_STORE = FAISSVectorStore(dim=dim)
        PARENT_STORE.add(PARENT_CHUNKS, p_embs)

    print(f'FAISS deposu hazir — {VECTOR_STORE.size} vektor, {dim} boyut')

## 🚀 5. Advanced RAG Teknikleri

| Teknik | Açıklama |
|---|---|
| **Naive RAG** | Sorguyu doğrudan göm, en yakın chunk'ı getir |
| **HyDE** | Önce varsayımsal cevap üret, onu göm |
| **Query Expansion** | Sorguyu çoğalt, RRF ile birleştir |
| **Reranking** | LLM ile getirilen chunk'ları yeniden sırala |
| **Parent-Child** | Küçük chunk bul, büyük parent'ı context olarak ver |
| **Hybrid Search** | Dense (FAISS) + Sparse (BM25) puanları birleştir |

In [ ]:
# ── HyDE: Hypothetical Document Embeddings ──────────────────
def hyde_retrieve(query: str, vector_store: FAISSVectorStore,
                  k: int = 5):
    hypo_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {'role': 'system', 'content': 'Kisa, olgusal bir paragraf yaz (~100 kelime).'},
            {'role': 'user',   'content': f'Soru: {query}'}
        ],
        max_tokens=200, temperature=0
    )
    hypothetical = hypo_resp.choices[0].message.content
    hypo_emb = get_embeddings([hypothetical])[0]
    results  = vector_store.search(hypo_emb, k=k)
    for r in results: r['technique'] = 'HyDE'
    return results, hypothetical


# ── Query Expansion + RRF ────────────────────────────────────
def query_expansion_retrieve(query: str, vector_store: FAISSVectorStore,
                              k: int = 5, n_expansions: int = 3):
    expand_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {'role': 'system', 'content': f'Soruyu {n_expansions} farklı sekilde ifade et. Her birini yeni satira yaz.'},
            {'role': 'user',   'content': query}
        ],
        max_tokens=300, temperature=0.3
    )
    expanded = [query] + [l.strip() for l in expand_resp.choices[0].message.content.split('\n') if l.strip()]
    rrf_scores: Dict[int, float] = {}
    all_chunks: Dict[int, Dict]  = {}
    for eq in expanded:
        emb     = get_embeddings([eq])[0]
        results = vector_store.search(emb, k=k)
        for rank, res in enumerate(results):
            cid = res['chunk_id']
            rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (60 + rank + 1)
            all_chunks[cid] = res
    ranked = sorted(rrf_scores.items(), key=lambda x: -x[1])[:k]
    results = []
    for cid, rrf in ranked:
        r = dict(all_chunks[cid])
        r['score'] = rrf; r['technique'] = 'QueryExpansion'
        results.append(r)
    return results, expanded


# ── LLM Reranking ────────────────────────────────────────────
def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    numbered = '\n\n'.join(f'[{i+1}] {c["text"][:400]}' for i, c in enumerate(candidates))
    resp = client.chat.completions.create(
        model=RERANK_MODEL,
        messages=[
            {'role': 'system', 'content': (
                f'Belgeleri soru icin alakaya gore sirala. En alakali {top_k} inin '
                'numarasini JSON listesi olarak dondur. Ornek: [2,1,4]. Sadece JSON ver.')},
            {'role': 'user',   'content': f'Soru: {query}\n\nBelgeler:\n{numbered}'}
        ],
        max_tokens=50, temperature=0
    )
    try:
        raw = resp.choices[0].message.content.strip()
        indices = json.loads(raw)
        reranked = [candidates[i-1] for i in indices if 1 <= i <= len(candidates)]
        for r in reranked: r['technique'] = 'Reranked'
        return reranked[:top_k]
    except Exception:
        return candidates[:top_k]


# ── Hybrid Search: FAISS + BM25 ──────────────────────────────
from rank_bm25 import BM25Okapi

def build_bm25(chunks: List[Dict]) -> BM25Okapi:
    return BM25Okapi([c['text'].lower().split() for c in chunks])

def hybrid_retrieve(query: str, vector_store: FAISSVectorStore,
                    bm25: BM25Okapi, chunks: List[Dict],
                    k: int = 5, alpha: float = 0.7) -> List[Dict]:
    q_emb   = get_embeddings([query])[0]
    dense   = vector_store.search(q_emb, k=len(chunks))
    dense_s = {r['chunk_id']: r['score'] for r in dense}
    bm25_raw = bm25.get_scores(query.lower().split())
    bm25_max = bm25_raw.max() + 1e-10
    combined = {}
    for i, c in enumerate(chunks):
        combined[c['chunk_id']] = (alpha * dense_s.get(c['chunk_id'], 0)
                                   + (1 - alpha) * bm25_raw[i] / bm25_max)
    ranked    = sorted(combined.items(), key=lambda x: -x[1])[:k]
    chunk_map = {c['chunk_id']: c for c in chunks}
    results   = []
    for cid, score in ranked:
        r = dict(chunk_map[cid])
        r['score'] = score; r['technique'] = 'Hybrid'
        results.append(r)
    return results


# ── Parent-Child Retrieval ────────────────────────────────────
def parent_child_retrieve(query: str, child_store: FAISSVectorStore,
                           parent_chunks: List[Dict], k: int = 3) -> List[Dict]:
    q_emb    = get_embeddings([query])[0]
    children = child_store.search(q_emb, k=k*2)
    seen, results = set(), []
    parent_map = {p['chunk_id']: p for p in parent_chunks}
    for child in children:
        pid = child.get('parent_id')
        if pid not in seen and pid in parent_map:
            seen.add(pid)
            r = dict(parent_map[pid])
            r['score'] = child['score']; r['technique'] = 'Parent-Child'
            results.append(r)
        if len(results) >= k: break
    return results


print('✅ Tum advanced retrieval fonksiyonlari hazir.')

## 🔗 6. Tam RAG Pipeline — İnteraktif Sorgulama

In [ ]:
def rag_answer(query: str, retrieved_chunks: List[Dict], technique: str = '') -> Dict:
    context = '\n\n---\n\n'.join(
        f'[Kaynak {i+1}]\n{c["text"][:800]}' for i, c in enumerate(retrieved_chunks)
    )
    system_prompt = (
        'Sen yardimci bir asistansin. Cevabini SADECE verilen baglamdan olustur.\n'
        'Baglamda olmayan bilgileri kesinlikle uydurma. Bilmiyorsan bunu belirt.\n'
        'Cevabinin sonunda kaynakları belirt ([Kaynak 1], [Kaynak 2] vb.).'
    )
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': f'Bagalm:\n{context}\n\nSoru: {query}'}
        ],
        max_tokens=600, temperature=0
    )
    return {'query': query, 'answer': resp.choices[0].message.content,
            'context': context, 'chunks': retrieved_chunks,
            'technique': technique, 'tokens': resp.usage.total_tokens}


q_input  = widgets.Text(placeholder='Soru girin...', layout=widgets.Layout(width='75%'))
tech_dd  = widgets.Dropdown(
    options=[('Naive (Temel)', 'naive'), ('HyDE', 'hyde'), ('Query Expansion', 'qe'),
             ('Hybrid Search (FAISS+BM25)', 'hybrid'), ('Reranking', 'rerank'),
             ('Parent-Child', 'parent_child')],
    description='Teknik:', style={'description_width': 'initial'}
)
k_slider = widgets.IntSlider(value=5, min=1, max=10, description='Top-K:',
                              style={'description_width': 'initial'})
ask_btn  = widgets.Button(description='Sor', button_style='primary', icon='search')
q_out    = widgets.Output()

RESULTS_LOG: List[Dict] = []

def on_ask(b):
    global RESULTS_LOG, BM25_INDEX
    with q_out:
        clear_output()
        if not VECTOR_STORE:
            print('Vektor deposu hazir degil.'); return
        query = q_input.value.strip()
        tech  = tech_dd.value
        k     = k_slider.value
        t0    = time.time()

        if tech == 'naive':
            emb    = get_embeddings([query])[0]
            chunks = VECTOR_STORE.search(emb, k=k)
        elif tech == 'hyde':
            chunks, hypo = hyde_retrieve(query, VECTOR_STORE, k=k)
            print(f'Varsayimsal cevap: {hypo[:200]}...')
        elif tech == 'qe':
            chunks, exp = query_expansion_retrieve(query, VECTOR_STORE, k=k)
            print(f'Genisletilmis sorgular: {exp}')
        elif tech == 'hybrid':
            if BM25_INDEX is None: BM25_INDEX = build_bm25(CHUNKS)
            chunks = hybrid_retrieve(query, VECTOR_STORE, BM25_INDEX, CHUNKS, k=k)
        elif tech == 'rerank':
            emb    = get_embeddings([query])[0]
            cands  = VECTOR_STORE.search(emb, k=k*2)
            chunks = rerank(query, cands, top_k=k)
        elif tech == 'parent_child':
            if not PARENT_CHUNKS:
                print('Parent-Child icin Parent-Child chunking secin.'); return
            chunks = parent_child_retrieve(query, VECTOR_STORE, PARENT_CHUNKS, k=k)

        result = rag_answer(query, chunks, technique=tech)
        result['latency'] = time.time() - t0
        RESULTS_LOG.append(result)

        print(f'Soru   : {query}')
        print(f'Teknik : {tech} | Sure: {result["latency"]:.1f}s | Token: {result["tokens"]}')
        print('-' * 60)
        print(result['answer'])
        print('\nGetirilen chunk lar:')
        for i, c in enumerate(chunks):
            print(f'  [{i+1}] score={c.get("score",0):.4f} | {c["text"][:100]}...')

ask_btn.on_click(on_ask)
display(widgets.VBox([
    widgets.HBox([q_input, ask_btn]),
    widgets.HBox([tech_dd, k_slider]),
    q_out
]))

## 📊 7. Performans Değerlendirmesi

In [ ]:
# Test soru seti — kendi sorularinizla degistirin
TEST_QUESTIONS = [
    'HyDE teknigi nasil calisir?',
    'Chunking stratejileri nelerdir?',
    'Hybrid search nedir?',
    'text-embedding-3-small modeli ne is yapar?',
    'Context precision ne anlama gelir?',
]

GROUND_TRUTHS = [
    'HyDE, sorguya varsayimsal bir cevap uretir ve o cevabin embeddingini retrieval icin kullanir.',
    'Fixed-size, recursive character split ve parent-child chunking baslica stratejilerdir.',
    'Hybrid search, dense vektor aramasi (FAISS) ile sparse BM25 keyword aramasini birlestir.',
    'text-embedding-3-small, metinleri 1536 boyutlu vektorlere donusturulen bir OpenAI embedding modelidir.',
    'Context precision, getirilen chunklarin soruyla ne kadar alakali oldugunu olcer.',
]

print(f'{len(TEST_QUESTIONS)} test sorusu hazir.')

In [ ]:
# Toplu degerlendirme — her teknik x her soru
EVAL_TECHNIQUES = ['naive', 'hyde', 'qe', 'hybrid', 'rerank']
EVAL_RESULTS: List[Dict] = []

if not VECTOR_STORE:
    print('Vektor deposu hazir degil.')
else:
    if BM25_INDEX is None: BM25_INDEX = build_bm25(CHUNKS)
    for tech in EVAL_TECHNIQUES:
        print(f'Teknik: {tech}')
        for q, gt in tqdm(zip(TEST_QUESTIONS, GROUND_TRUTHS),
                          total=len(TEST_QUESTIONS), desc=tech):
            t0 = time.time()
            try:
                if tech == 'naive':
                    emb = get_embeddings([q])[0]
                    chunks = VECTOR_STORE.search(emb, k=5)
                elif tech == 'hyde':
                    chunks, _ = hyde_retrieve(q, VECTOR_STORE, k=5)
                elif tech == 'qe':
                    chunks, _ = query_expansion_retrieve(q, VECTOR_STORE, k=5)
                elif tech == 'hybrid':
                    chunks = hybrid_retrieve(q, VECTOR_STORE, BM25_INDEX, CHUNKS, k=5)
                elif tech == 'rerank':
                    emb   = get_embeddings([q])[0]
                    cands = VECTOR_STORE.search(emb, k=10)
                    chunks = rerank(q, cands, top_k=5)
                result = rag_answer(q, chunks, technique=tech)
                result['ground_truth'] = gt
                result['latency']      = time.time() - t0
                EVAL_RESULTS.append(result)
            except Exception as e:
                print(f'  Hata ({q[:30]}): {e}')

    print(f'\nToplam {len(EVAL_RESULTS)} sonuc toplandi.')

In [ ]:
# LLM-as-Judge: 4 metrik 0-1 arasi puanlama
def llm_judge(question: str, answer: str, context: str, ground_truth: str) -> Dict:
    prompt = (
        'RAG ciktisin degerlendir. 4 metrigi 0.0-1.0 arasi puanla. Sadece JSON dondur.\n\n'
        f'Soru: {question}\n'
        f'Cevap: {answer}\n'
        f'Bagalm (ilk 800 karakter): {context[:800]}\n'
        f'Referans cevap: {ground_truth}\n\n'
        'Metrikler:\n'
        '- faithfulness: Cevap baglamla ne kadar tutarli? (uydurma var mi?)\n'
        '- answer_relevancy: Cevap soruyu ne kadar iyi yanitliyor?\n'
        '- context_precision: Bagalm soruyla ne kadar alakali?\n'
        '- completeness: Referans cevaba kiyasla ne kadar eksiksiz?\n\n'
        'JSON: {"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0, "completeness": 0.0}'
    )
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=100, temperature=0
    )
    try:
        raw = re.sub(r'```json|```', '', resp.choices[0].message.content).strip()
        return json.loads(raw)
    except Exception:
        return {'faithfulness': 0.5, 'answer_relevancy': 0.5,
                'context_precision': 0.5, 'completeness': 0.5}


print('LLM-as-Judge skorlari hesaplaniyor...')
for res in tqdm(EVAL_RESULTS, desc='Judge'):
    scores = llm_judge(res['query'], res['answer'], res['context'], res['ground_truth'])
    res.update(scores)

print('Puanlama tamamlandi.')

In [ ]:
METRICS = ['faithfulness', 'answer_relevancy', 'context_precision', 'completeness']

if not EVAL_RESULTS:
    print("❌ Hata: EVAL_RESULTS bos. Lutfen once dokuman yukleme, chunking ve degerlendirme adimlarini calistirin.")
    # Boş bir özet DataFrame oluşturarak sonraki hücrelerde olası hataları önle
    summary = pd.DataFrame(columns=['faithfulness', 'answer_relevancy', 'context_precision',
                                    'completeness', 'avg_latency', 'avg_tokens', 'composite_score'])
else:
    df = pd.DataFrame([
        {'technique': r['technique'], 'query': r['query'][:40] + '...',
         'latency': round(r['latency'], 2), 'tokens': r['tokens'],
         **{m: r.get(m, 0.5) for m in METRICS}}
        for r in EVAL_RESULTS
    ])

    summary = df.groupby('technique').agg(
        faithfulness     =('faithfulness',      'mean'),
        answer_relevancy =('answer_relevancy',  'mean'),
        context_precision=('context_precision', 'mean'),
        completeness     =('completeness',      'mean'),
        avg_latency      =('latency',           'mean'),
        avg_tokens       =('tokens',            'mean'),
    ).round(3)

    summary['composite_score'] = summary[METRICS].mean(axis=1).round(3)
    summary = summary.sort_values('composite_score', ascending=False)

    print('Teknik bazli ozet:')
    display(summary)

## 📈 Görsel Analiz

In [ ]:
techs  = summary.index.tolist()
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Advanced RAG — Performans Degerlendirmesi', fontsize=14, fontweight='bold')

# Grouped Bar
ax1 = axes[0]
x = np.arange(len(METRICS))
width = 0.15
for i, (tech, color) in enumerate(zip(techs, colors)):
    vals = summary.loc[tech, METRICS].values
    ax1.bar(x + i*width, vals, width, label=tech, color=color, alpha=0.85)
ax1.set_xticks(x + width*(len(techs)-1)/2)
ax1.set_xticklabels([m.replace('_','\n') for m in METRICS], fontsize=9)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Skor (0-1)')
ax1.set_title('Metrik Bazli Karsilastirma')
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)
ax1.axhline(0.7, color='gray', linestyle='--', alpha=0.5)

# Latency vs Quality Scatter
ax2 = axes[1]
for i, (tech, color) in enumerate(zip(techs, colors)):
    row = summary.loc[tech]
    ax2.scatter(row['avg_latency'], row['composite_score'],
                s=row['avg_tokens']/5, color=color, alpha=0.8,
                edgecolors='white', linewidth=1.5, label=tech)
    ax2.annotate(tech, (row['avg_latency']+0.05, row['composite_score']+0.01), fontsize=8)
ax2.set_xlabel('Gecikme (sn)')
ax2.set_ylabel('Kompozit Skor')
ax2.set_title('Hiz vs Kalite (baloncuk = token kullanimi)')
ax2.grid(alpha=0.3)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/rag_eval_1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
ax1 = axes[0]
heat_data = summary[METRICS + ['composite_score']].copy()
sns.heatmap(heat_data, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, vmax=1,
            ax=ax1, linewidths=0.5)
ax1.set_title('Teknik x Metrik Isi Haritasi')
ax1.set_ylabel('')

# Radar Chart
ax2 = plt.subplot(122, polar=True)
angles = np.linspace(0, 2*np.pi, len(METRICS), endpoint=False).tolist()
angles += angles[:1]
for i, (tech, color) in enumerate(zip(techs, colors)):
    vals = summary.loc[tech, METRICS].tolist() + [summary.loc[tech, METRICS[0]]]
    ax2.plot(angles, vals, color=color, linewidth=2, label=tech)
    ax2.fill(angles, vals, color=color, alpha=0.08)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels([m.replace('_','\n') for m in METRICS], size=9)
ax2.set_ylim(0, 1)
ax2.set_title('Radar Karsilastirmasi', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/rag_eval_2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Soru bazli detayli tablo
fig, ax = plt.subplots(figsize=(18, max(4, len(df)*0.45 + 1)))
ax.axis('off')

cols_show = ['technique', 'query', 'faithfulness', 'answer_relevancy',
             'context_precision', 'completeness', 'latency']
tbl_data  = df[cols_show].round(3).values.tolist()
tbl_cols  = ['Teknik', 'Soru', 'Faith.', 'Rel.', 'Prec.', 'Comp.', 'Sure(s)']

tbl = ax.table(cellText=tbl_data, colLabels=tbl_cols, cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.4)

metric_cols = [2, 3, 4, 5]
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2C3E50')
        cell.set_text_props(color='white', fontweight='bold')
    elif col in metric_cols:
        try:
            val = float(cell.get_text().get_text())
            cell.set_facecolor(plt.cm.RdYlGn(val))
        except ValueError:
            pass

ax.set_title('Soru Bazli Degerlendirme Tablosu', fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('/tmp/rag_eval_3.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gecikme ve token kullanim analizi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
df.boxplot(column='latency', by='technique', ax=ax1, patch_artist=True)
ax1.set_title('Gecikme Dagilimi (sn)')
ax1.set_xlabel('Teknik')
ax1.set_ylabel('Gecikme (sn)')
plt.sca(ax1)
plt.xticks(rotation=30, ha='right')

ax2 = axes[1]
token_summary = df.groupby('technique')['tokens'].mean().sort_values()
bars = ax2.barh(token_summary.index, token_summary.values,
                color=colors[:len(token_summary)], alpha=0.8)
ax2.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax2.set_title('Ortalama Token Kullanimi')
ax2.set_xlabel('Token sayisi')
ax2.grid(axis='x', alpha=0.3)

fig.suptitle('')
plt.tight_layout()
plt.savefig('/tmp/rag_eval_4.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_tech  = summary['composite_score'].idxmax()
best_score = summary.loc[best_tech, 'composite_score']
fastest    = summary['avg_latency'].idxmin()

print('=' * 65)
print('  ADVANCED RAG OZET RAPORU')
print('=' * 65)
print(f'  En yuksek kalite : {best_tech:<20} (skor={best_score:.3f})')
print(f'  En hizli teknik  : {fastest}')
print()
print('  Teknik Secim Rehberi:')
print('  ' + '-'*32)
print('  Naive        -> Hizli prototip, dusuk maliyet')
print('  HyDE         -> Kisa veya belirsiz sorgular')
print('  QueryExpand  -> Genis konu kapsamı gerektiginde')
print('  Hybrid       -> Hem anahtar kelime hem anlam onemli')
print('  Reranking    -> Hassasiyet oncelikli, gecikme ikincil')
print('  Parent-Child -> Uzun belgeler, genis bagalm gereksinimi')
print()
print('  Gorseller /tmp/rag_eval_*.png olarak kaydedildi.')
print('=' * 65)